# Performance and Capacity of Data Pipelines

Notebook for **Chapter 14** of the *de* study document. Run the cells top to bottom. Read the full explanation on the site: https://dadipavan.github.io/Learning/study/de/ch14.html

Tip: change a number, predict the output, then run the cell again.

## 14.1 Little's Law, Utilization and Latency: the Math Behind Every Test

In [ ]:
import random, statistics
random.seed(5)
def sim(arrival_rate, service_rate, n=200000):
    t=0; free_at=0; waits=[]
    for _ in range(n):
        t+=random.expovariate(arrival_rate)
        start=max(t,free_at); s=random.expovariate(service_rate); free_at=start+s
        waits.append(free_at-t)    # latency = wait + service
    return waits
mu=100.0
for rho in (0.5,0.7,0.8,0.9,0.95):
    w=sim(rho*mu,mu); w.sort()
    mean=statistics.mean(w); p95=w[int(.95*len(w))]; p99=w[int(.99*len(w))]
    print(f"{rho:.2f}  {mean*1000:10.1f}  {p95*1000:6.1f}  {p99*1000:6.1f}   {rho*mu*mean:6.2f}")
print("theory mean M/M/1: 1/(mu-lambda) ->", [round(1000/(mu-r*mu),1) for r in (0.5,0.7,0.8,0.9,0.95)])

## 14.2 Load-Testing Data Pipelines, Kafka Consumers and ML Endpoints

In [ ]:
stages={"producer":8000,"kafka_broker":50000,"consumer(4 threads x 1200)":4*1200,"transform":6000,"warehouse_sink":3500}
b=min(stages,key=stages.get); print("bottleneck:",b,stages[b])
arr=4000
print("arrival",arr,"util at bottleneck",round(arr/stages[b],2))

bottleneck: warehouse_sink 3500
arrival 4000 util at bottleneck 1.14


In [ ]:
node=0.9; rps_node=150  # illustrative price per hour, measured requests per second per node
print("cost per 1k predictions $",round(node/(rps_node*3600)*1000,5))

cost per 1k predictions $ 0.00167


In [ ]:
import random, statistics as st
random.seed(11)
base=[random.gauss(250,12) for _ in range(30)]
new=[random.gauss(280,12) for _ in range(6)]
print("baseline mean %.1f sd %.1f"%(st.mean(base),st.stdev(base)),"| new mean %.1f"%st.mean(new))
diff=st.mean(new)-st.mean(base)
cnt=0; N=10000
for _ in range(N):
    s=[random.choice(base) for _ in range(6)]     # bootstrap: 6 runs drawn from history
    if st.mean(s)-st.mean(base)>=diff: cnt+=1
print("observed diff %.1f ms; bootstrap p = %.4f"%(diff,cnt/N))

baseline mean 250.8 sd 11.7 | new mean 272.7
observed diff 21.9 ms; bootstrap p = 0.0000


## 14.3 Capstone Projects

Each project is a portfolio piece. Do them in this order; each one gives material for interview answers. For every project write a README with the problem, architecture diagram, how to run, results with real numbers, and what you would change. Use free tools (Docker, PostgreSQL, DuckDB, Airflow, Kafka in Docker, Spark local, dbt with DuckDB).

## 14.4 The Data Engineering Interview Bank

Answer aloud first, then compare. Model answers are short on purpose; add an example from your own work. They cover the whole document. Chapter numbers show where to review.

## 14.5 Sizing Examples: the Latency Curve, Fast versus Many Servers, and Burst Lag

Section 14.1 gave you Little's law and a table of the utilization-latency curve. Here is the picture of that curve, then two more capacity questions you will be asked in design reviews: "should I run one big consumer or many small ones?" and "how long will the lag last after a traffic burst?".

In [ ]:
# One fast server versus two servers each half as fast, same total capacity, same load.
import random, statistics
def sim(lam, mu_each, servers, n=100000, seed=11):
    random.seed(seed)
    t = 0.0; free = [0.0] * servers; lat = []
    for _ in range(n):
        t += random.expovariate(lam)
        i = free.index(min(free)); start = max(t, free[i])
        free[i] = start + random.expovariate(mu_each)
        lat.append(free[i] - t)
    lat.sort()
    return statistics.mean(lat) * 1000, lat[int(0.99 * n)] * 1000
lam = 80.0     # messages per second, total capacity 100 per second in both setups
m1, p1 = sim(lam, 100.0, 1)
m2, p2 = sim(lam, 50.0, 2)
m4, p4 = sim(lam, 25.0, 4)
print("1 server  at 100/s   : mean %6.1f ms  p99 %6.1f ms" % (m1, p1))
print("2 servers at 50/s each: mean %6.1f ms  p99 %6.1f ms" % (m2, p2))
print("4 servers at 25/s each: mean %6.1f ms  p99 %6.1f ms" % (m4, p4))

1 server  at 100/s   : mean   50.2 ms  p99  227.1 ms
2 servers at 50/s each: mean   55.9 ms  p99  237.8 ms
4 servers at 25/s each: mean   70.3 ms  p99  276.0 ms


In [ ]:
# A burst: input is 150 msg/s for 60 s on a consumer group that handles 100 msg/s, then back to 60 msg/s.
cap = 100
lag, peak, t_clear = 0, 0, None
for sec in range(0, 600):
    arrive = 150 if sec < 60 else 60
    lag = max(0, lag + arrive - cap)
    peak = max(peak, lag)
    if sec >= 60 and lag == 0 and t_clear is None: t_clear = sec
print("peak lag:", peak, "messages")
print("lag cleared at second:", t_clear)
print("catch-up rate after the burst:", cap - 60, "msg/s")
print("peak lag in seconds of work:", peak / cap, "s")

peak lag: 3000 messages
lag cleared at second: 134
catch-up rate after the burst: 40 msg/s
peak lag in seconds of work: 30.0 s


## Chapter recap

## Practice set

**1.** (easy) 500 requests/s with 80 ms latency. How many requests in flight?

## Prompt pack

---
Done? Do the **Practice set** at the end of the chapter, then clear your Review cards on the site.

https://dadipavan.github.io/Learning/study/de/ch14.html